# 🔬 Retail Sales ML Pipeline

This notebook walks through the **entire ML pipeline** end-to-end:

1. Schema definition
2. Synthetic data generation
3. Exploratory data analysis
4. Feature engineering
5. Train / test split
6. Baseline forecasting
7. Baseline evaluation
8. XGBoost model
9. Full model comparison
10. Visualization & insights

Each section explains **why** the step matters, not just how.

## 1 · Setup & Imports

We import the project's own modules so every step uses the same schema and configuration objects. No code is duplicated from source.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

# Ensure the project source is importable
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project modules
from retail_ts_ml.data.schema import ForecastSchema, build_column_role_frame
from retail_ts_ml.data.synthetic import generate_synthetic_sales_data
from retail_ts_ml.config.settings import SyntheticDataConfig, FeatureConfig
from retail_ts_ml.features.time_series import build_time_series_features, prepare_supervised_frame
from retail_ts_ml.models.baselines import build_baseline_forecast_frame
from retail_ts_ml.evaluation.backtesting import (
    temporal_holdout_split, generate_rolling_origin_splits,
    evaluate_forecast_frame, RollingWindowSpec,
)
from retail_ts_ml.visualization.summary import plot_retailer_trends, plot_forecast_vs_actual

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
print('✅ All imports successful')

## 2 · Schema Definition

The `ForecastSchema` class is the **single source of truth** for column names.
Every module references it instead of hard-coding strings. This means when real data arrives, we only update the schema — nothing else changes.

In [ ]:
schema = ForecastSchema()
print(f'Time key:      {schema.time_key}')
print(f'Entity keys:   {schema.entity_keys}')
print(f'Target:        {schema.target}')
print(f'Frequency:     {schema.frequency}')
print(f'All columns:   {len(schema.all_columns())}')
print()

# Column role dictionary
role_frame = build_column_role_frame(schema)
role_frame

## 3 · Synthetic Data Generation

We generate **~14,500 rows** of weekly retail sales data across 4 retailers × 24 products × 156 weeks.
The generator injects realistic features:
- **Trend**: some series grow, some decline
- **Seasonality**: holiday and summer spikes
- **Promotions**: random promo events with configurable lift
- **Anomalies**: rare demand shocks
- **Missingness**: realistic gaps in price/inventory

> **Why synthetic?** It lets us validate the full pipeline before real data lands. The schema is identical, so transitioning is a config change.

In [ ]:
config = SyntheticDataConfig()
df = generate_synthetic_sales_data(config=config, schema=schema)

print(f'Shape: {df.shape}')
print(f'Date range: {df[schema.time_key].min().date()} → {df[schema.time_key].max().date()}')
print(f'Retailers: {df[schema.entity_keys[0]].nunique()}')
print(f'Products:  {df[schema.entity_keys[1]].nunique()}')
print(f'Missing %:')
print((df.isnull().sum() / len(df) * 100).round(2).to_string())
print()
df.head(10)

## 4 · Exploratory Data Analysis

Before any modeling, we need to understand:
1. **Distribution** of the target variable — skew affects model choice
2. **Trend** per retailer — are sales growing or declining?
3. **Seasonality** — are there repeating weekly/monthly patterns?
4. **Outliers** — extreme values that could dominate RMSE

In [ ]:
# 4a — Target distribution
fig = px.histogram(
    df, x=schema.target, nbins=80,
    title='Distribution of Target Variable (units_sold)',
    color_discrete_sequence=['#636EFA'],
    opacity=0.85,
)
fig.update_layout(template='plotly_white', height=400)
fig.show()

print(f'Skewness: {df[schema.target].skew():.2f}')
print(f'Kurtosis: {df[schema.target].kurtosis():.2f}')
print(f'Mean:     {df[schema.target].mean():.1f}')
print(f'Median:   {df[schema.target].median():.1f}')

In [ ]:
# 4b — Retailer trend overview (uses project viz module)
fig = plot_retailer_trends(df, schema=schema)
fig.update_layout(template='plotly_white', height=500)
fig.show()

In [ ]:
# 4c — Weekly sales heatmap by retailer
pivot = (
    df.assign(week=df[schema.time_key].dt.isocalendar().week.astype(int))
    .groupby([schema.entity_keys[0], 'week'])[schema.target]
    .mean()
    .reset_index()
    .pivot(index=schema.entity_keys[0], columns='week', values=schema.target)
)
fig = px.imshow(
    pivot, aspect='auto',
    title='Average Weekly Sales by Retailer (Seasonality Check)',
    color_continuous_scale='Viridis',
    labels={'color': 'Avg Sales'},
)
fig.update_layout(template='plotly_white', height=350)
fig.show()

In [ ]:
# 4d — Box plot for outlier detection
fig = px.box(
    df, x=schema.entity_keys[0], y=schema.target,
    title='Sales Distribution by Retailer (Outlier Check)',
    color=schema.entity_keys[0],
)
fig.update_layout(template='plotly_white', height=450, showlegend=False)
fig.show()

## 5 · Feature Engineering

Tree-based models **cannot extrapolate** — they have no concept of "next week".
We make time information explicit through:

| Feature Type | Examples | Why |
|---|---|---|
| **Lag features** | lag_1, lag_4, lag_52 | Capture autocorrelation |
| **Rolling stats** | rolling_mean_4, rolling_std_12 | Capture local trend & volatility |
| **Calendar** | month, quarter, week_of_year | Capture seasonality |
| **Share** | object_share_within_retailer | Capture relative position |

> ⚠️ All lag/rolling features use `.shift(1)` to prevent **data leakage** — we never let the model see the current target when computing features.

In [ ]:
feature_config = FeatureConfig()
print(f'Lag periods:      {feature_config.lags}')
print(f'Rolling windows:  {feature_config.rolling_windows}')
print()

# Build supervised frame (drops rows with NaN lags)
supervised = prepare_supervised_frame(df, schema=schema, config=feature_config)

print(f'Original rows:    {len(df):,}')
print(f'Supervised rows:  {len(supervised):,}')
print(f'Columns added:    {len(supervised.columns) - len(df.columns)}')
print()

# Show new feature columns
new_cols = [c for c in supervised.columns if c not in df.columns]
print('New features:', new_cols)
print()
supervised[new_cols].describe().round(2)

## 6 · Train / Test Split

**Critical rule**: For time series, we **never** randomly shuffle. The split must be temporal — train on the past, test on the future.

We use `temporal_holdout_split()` with a 12-week horizon, meaning the last 12 weeks become the test set.

In [ ]:
train_df, test_df = temporal_holdout_split(df, schema=schema, horizon=12)

print(f'Train: {len(train_df):,} rows  |  '      f'{train_df[schema.time_key].min().date()} → {train_df[schema.time_key].max().date()}')
print(f'Test:  {len(test_df):,} rows  |  '      f'{test_df[schema.time_key].min().date()} → {test_df[schema.time_key].max().date()}')
print(f'\nTrain/Test ratio: {len(train_df)/len(df)*100:.1f}% / {len(test_df)/len(df)*100:.1f}%')

## 7 · Baseline Forecasting

**Any model that can't beat these baselines has negative value.**

| Model | Logic | When It Wins |
|---|---|---|
| **Naive** | Repeat last value | Stable, low-variance series |
| **Seasonal Naive** | Repeat same week last year | Strong annual seasonality |
| **Moving Average** | Average of last 4 values | Noisy but trendless series |

In [ ]:
baseline_forecasts = build_baseline_forecast_frame(
    train_df, schema=schema, horizon=12,
    season_length=52, moving_average_window=4,
)

print(f'Forecast rows: {len(baseline_forecasts):,}')
print(f'Models: {baseline_forecasts["model_name"].unique().tolist()}')
print()
baseline_forecasts.head(12)

## 8 · Baseline Evaluation

We evaluate using **multiple metrics** because no single metric tells the full story:

| Metric | Measures | Good For |
|---|---|---|
| **MAE** | Average absolute error | Interpretable, same units as target |
| **RMSE** | Root mean squared error | Penalizes large errors more |
| **sMAPE** | Symmetric % error | Scale-independent comparison |
| **WAPE** | Weighted absolute % error | Volume-weighted accuracy |
| **Bias** | Systematic over/under | Detects systematic prediction drift |

In [ ]:
baseline_metrics = evaluate_forecast_frame(
    baseline_forecasts, test_df, schema=schema,
)

print('=== Baseline Model Comparison ===')
baseline_metrics

In [ ]:
# Visualize baseline metrics
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE', 'RMSE', 'sMAPE'])

colors = {'naive': '#636EFA', 'seasonal_naive': '#EF553B', 'moving_average': '#00CC96'}

for i, metric in enumerate(['mae', 'rmse', 'smape'], 1):
    for _, row in baseline_metrics.iterrows():
        fig.add_trace(
            go.Bar(
                x=[row['model_name']], y=[row[metric]],
                name=row['model_name'],
                marker_color=colors.get(row['model_name'], '#AB63FA'),
                showlegend=(i == 1),
            ),
            row=1, col=i,
        )

fig.update_layout(
    template='plotly_white', height=400,
    title_text='Baseline Model Metrics',
    barmode='group',
)
fig.show()

## 9 · XGBoost Model

**Why XGBoost over ARIMA?**
- Handles **multiple series** simultaneously (cross-learning)
- Handles **missing values** natively
- Captures **non-linear feature interactions** (promo × seasonality)
- Works well on **tabular data** — consistently outperforms neural nets under ~10K samples per series

We train on the supervised frame (with lag/rolling features) and predict on the test period.

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prepare supervised frames for train and test
train_supervised = prepare_supervised_frame(train_df, schema=schema, config=feature_config)
test_supervised = prepare_supervised_frame(
    pd.concat([train_df, test_df], ignore_index=True),  # need history for lags
    schema=schema, config=feature_config,
)
# Only keep rows that are in the test period
test_cutoff = train_df[schema.time_key].max()
test_supervised = test_supervised[test_supervised[schema.time_key] > test_cutoff].copy()

# Define feature columns (exclude target, keys, dates, non-numeric)
exclude = {schema.time_key, schema.target, schema.revenue_col, schema.margin_col,
           *schema.entity_keys, *schema.hierarchy_fields, 'channel'}
feature_cols = [c for c in train_supervised.columns
                if c not in exclude and train_supervised[c].dtype in ('float64', 'int64', 'int32')]

print(f'Features: {len(feature_cols)}')
print(f'Train rows: {len(train_supervised):,}')
print(f'Test rows:  {len(test_supervised):,}')
print()

X_train = train_supervised[feature_cols]
y_train = train_supervised[schema.target]
X_test = test_supervised[feature_cols]
y_test = test_supervised[schema.target]

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

preds = model.predict(X_test)
preds = np.maximum(preds, 0)  # clip negatives

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f'XGBoost MAE:  {mae:.2f}')
print(f'XGBoost RMSE: {rmse:.2f}')

In [ ]:
# Feature importance (top 15)
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
top15 = importance.tail(15)

fig = px.bar(
    x=top15.values, y=top15.index,
    orientation='h',
    title='Top 15 Feature Importances (XGBoost)',
    labels={'x': 'Importance', 'y': 'Feature'},
    color_discrete_sequence=['#636EFA'],
)
fig.update_layout(template='plotly_white', height=500)
fig.show()

## 10 · Full Model Comparison

We combine all results into a single comparison table and chart.
The best model isn't always the most complex — if a simple baseline performs comparably, it's often preferred for interpretability and robustness.

In [ ]:
# Build XGBoost forecast frame to match baseline format
xgb_forecast = test_supervised[[schema.time_key, *schema.entity_keys]].copy()
xgb_forecast['prediction'] = preds
xgb_forecast['model_name'] = 'xgboost'

# Evaluate XGBoost with the same function
xgb_metrics = evaluate_forecast_frame(xgb_forecast, test_df, schema=schema)

# Combine all metrics
all_metrics = pd.concat([baseline_metrics, xgb_metrics], ignore_index=True)
all_metrics = all_metrics.sort_values('mae').reset_index(drop=True)

print('=== Full Model Comparison ===')
all_metrics

In [ ]:
# Final comparison chart
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['MAE (lower is better)', 'RMSE (lower is better)'],
)

model_colors = {
    'naive': '#636EFA', 'seasonal_naive': '#EF553B',
    'moving_average': '#00CC96', 'xgboost': '#AB63FA',
}

for i, metric in enumerate(['mae', 'rmse'], 1):
    sorted_df = all_metrics.sort_values(metric)
    fig.add_trace(
        go.Bar(
            x=sorted_df['model_name'],
            y=sorted_df[metric],
            marker_color=[model_colors.get(m, '#FFA15A') for m in sorted_df['model_name']],
            showlegend=False,
        ),
        row=1, col=i,
    )

fig.update_layout(
    template='plotly_white', height=450,
    title_text='All Models — MAE & RMSE Side by Side',
)
fig.show()

## ✅ Pipeline Complete

### What we demonstrated:
- Schema-driven data generation with realistic retail patterns
- Proper temporal train/test splitting (no data leakage)
- Lag and rolling features for tree-based models
- Baseline-first approach before advanced models
- Multi-metric evaluation for comprehensive comparison

### Next steps:
- [ ] Plug in real data via `database_profiles.py`
- [ ] Add hyperparameter tuning with Optuna
- [ ] Rolling-origin backtesting for robustness
- [ ] Per-retailer / per-category model comparison
- [ ] Error decomposition by segment